In [1]:
import pandas as pd
import numpy as np
import mlflow.sklearn
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTEENN
import optuna
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

c:\Users\iuri_\OneDrive\Desktop\youtube_sentiment_mlops_pipeline\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("reddit_preprocessed.csv")
df.head()

,clean_comment,category,words,stop_words,characters,punctuation_chars
0,family mormon never tried explain still stare ...,1,39,13,259,0
1,buddhism much lot compatible christianity espe...,1,196,59,1268,0
2,seriously say thing first get complex explain ...,-1,86,40,459,0
3,learned want teach different focus goal not wr...,0,29,15,167,0
4,benefit may want read living buddha living chr...,1,112,45,690,0


In [6]:
! pip install dotenv sentence-transformers

  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import mlflow
import os 
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()
mlflow_url = os.environ.get("MLFLOW_URL")

mlflow.set_tracking_uri(mlflow_url)


In [9]:
mlflow.set_experiment("Testing sbert embedding with hyperparameter tuning in lightgbm model")

2025/11/15 18:53:09 INFO mlflow.tracking.fluent: Experiment with name 'Testing sbert embedding with hyperparameter tuning in lightgbm model' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-bucket-078/237401474257940949', creation_time=1763243588601, experiment_id='237401474257940949', last_update_time=1763243588601, lifecycle_stage='active', name='Testing sbert embedding with hyperparameter tuning in lightgbm model', tags={}>

In [11]:
# -----------------

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=0)

X_train = sbert_model.encode(X_train.tolist())
X_test = sbert_model.encode(X_test.tolist()) # correct way to do it, without data leakage

rus = RandomUnderSampler(random_state=0) 
X_train, y_train = rus.fit_resample(X_train, y_train)
# -----------------

# Function to log results in MLflow
def log_bestmodel_mlflow(model_name, model, X_train, X_test, y_train, y_test):

    with mlflow.start_run():

        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_undersampling_sbert_embedding")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=0, verbosity=-1)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    best_params = study.best_params
    best_model = lgb.LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=0, verbosity=-1)

    log_bestmodel_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)

    return best_params



In [12]:
best_params = run_optuna_experiment()

[I 2025-11-15 19:56:44,263] A new study created in memory with name: no-name-557ac432-6444-4077-bd21-71a30fce00c7
c:\Users\iuri_\OneDrive\Desktop\youtube_sentiment_mlops_pipeline\myvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-11-15 19:57:09,769] Trial 0 finished with value: 0.6453336987803207 and parameters: {'n_estimators': 126, 'learning_rate': 0.05478458374393853, 'max_depth': 6}. Best is trial 0 with value: 0.6453336987803207.
c:\Users\iuri_\OneDrive\Desktop\youtube_sentiment_mlops_pipeline\myvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-11-15 19:57:24,290] Trial 1 finished with value: 0.6287515417294779 and parameters: {'n_estimators': 288, 'learning_rate': 0.021784458789554866, 'max_depth': 4}. Best is trial 0 wit

🏃 View run LightGBM_undersampling_sbert_embedding at: http://ec2-18-222-172-50.us-east-2.compute.amazonaws.com:5000/#/experiments/237401474257940949/runs/78d9fb37fc34482eaedff0ec40eb0560
🧪 View experiment at: http://ec2-18-222-172-50.us-east-2.compute.amazonaws.com:5000/#/experiments/237401474257940949


In [13]:
best_params

{'n_estimators': 241, 'learning_rate': 0.09440051252026231, 'max_depth': 7}